In [1]:
import numpy as np
import jax
import jax.numpy as jnp
import esm2quinox
import pickle
import equinox as eqx

from jaxtyping import Array, Float, Int, PyTree

In [2]:
from flax import nnx

In [2]:

import importlib, inspect, sys
import jax, jaxlib
print("jax:", jax.__version__)
print("jaxlib:", jaxlib.__version__)
print("jaxlib file:", inspect.getfile(jaxlib))
# show files in jaxlib package dir
import os
print("jaxlib dir:", os.path.dirname(inspect.getfile(jaxlib)))

jax: 0.8.1
jaxlib: 0.8.1
jaxlib file: /home/kunzj/venv/esm2/lib/python3.12/site-packages/jaxlib/__init__.py
jaxlib dir: /home/kunzj/venv/esm2/lib/python3.12/site-packages/jaxlib


In [5]:
class CNN(eqx.Module):
    layers: list

    def __init__(self, key):
        key1, key2, key3, key4 = jax.random.split(key, 4)
        # Standard CNN setup: convolutional layer, followed by flattening,
        # with a small MLP on top.
        self.layers = [
            eqx.nn.Conv2d(1, 3, kernel_size=4, key=key1),
            eqx.nn.MaxPool2d(kernel_size=2),
            jax.nn.relu,
            jnp.ravel,
            eqx.nn.Linear(1728, 512, key=key2),
            jax.nn.sigmoid,
            eqx.nn.Linear(512, 64, key=key3),
            jax.nn.relu,
            eqx.nn.Linear(64, 10, key=key4),
            jax.nn.log_softmax,
        ]

    def __call__(self, x: Float[Array, "1 28 28"]) -> Float[Array, "10"]:
        for layer in self.layers:
            x = layer(x)
        return x


key, subkey = jax.random.split(key, 2)
model = CNN(subkey)